# Free-Droid (Szabi) — v14 fine-tune (CSAK 8B: „vegyes kérés" kategória + `lora_r` vissza 8-ra)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

## Mi ez a kör

**1. A `lora_r` VISSZA 8-ra — vagyis nem adunk át semmit.** A `--preset gentle` alapértéke
`lora_r=8, lora_alpha=8`, tehát a v13 `--lora-r 16 --lora-alpha 16` kapcsolópárja egyszerűen
lekerül a parancsról. Ez nem óvatosság, hanem mérés: a v13 egyváltozós köre pontosan ezt az egy
dolgot vitte (az `alpha` VELE EGYÜTT ment, tehát az `alpha/r` skálázás 1,0 maradt — a változás
tisztán kapacitás volt), és **mérhetően rosszabb lett**:

| | v12 (r=8) | v13 (r=16) |
| :-- | --: | --: |
| red_team arány | **72%** | 57% |
| `nyelv` bukás-címke | **3** | 6 |
| `mozgas_biztonsag` | **3/5** | 1/5 |
| `persona_provokacio` | **3/5** | 1/5 |

A kapacitás-emelés tehát nem alultanulást oldott fel, hanem rontott. Visszaáll a `gentle`.

**2. Új dataset: 1070 → 1098 példa** — a **vegyes kérés** kategória (28 példa,
`dataset/_build_mixed_requests.py`). A split most **988 / 110**. Ez is mérés, nem ötlet: a
2026-08-11-i red_team-en a bukott `mozgas_biztonsag` és `wifi_invarians` próbák **mindegyike**
ugyanaz a szerkezet — *részben jogos, részben tiltott* kérés —, és ezt a datasetben **0 példa
tanította** (22 összetett instruction: 14 végig jogos, 3 végig tiltott, 0 vegyes). A modell tudja,
hogy elhárítson, és tudja, hogy tool-t adjon; azt nem tudja, hogy EGY kérésen belül tegye mindkettőt
(a v12 elhárított szóban, majd kiadott egy `move forward full_speed`-et — csak a parser érvénytelen
értéke mentette meg).

**3. Epoch-jelöltek egyetlen bérlésből:** `--epochs 3`, `save_strategy="epoch"`, és a
`load_best_model_at_end` már be van építve a `finetune.py`-ba (PR #55) — a záró GGUF-export így
magától a legjobb `eval_loss`-ú epochot kapja, nem az utolsót (a v13-on ez pont visszafelé sült el:
e2=1,406 → e3=1,424, és a kirakott modell a fordulat rossz oldalán volt).

### Két dolog mozdul — és miért nem sérti ez az egyváltozós elvet

Nem hallgatjuk el: a dataset ÉS a `lora_r` is változik. Azért érvényes a kör, mert **a műszereik
diszjunktak**:

| Változó | Műszer |
| :-- | :-- |
| új „vegyes kérés" kategória | red_team `mozgas_biztonsag` + `wifi_invarians` (5+5 próba) — pont ez a két dimenzió bukott a vegyes alakon |
| `lora_r` 16 → 8 | a teljes red_team arány (40 próba) + a `nyelv` bukás-címke száma, plusz a gépi `nyelv_probe.py` |

A `lora_r` visszaállítása ráadásul **nem új beállítás**: a v12 pontosan ez volt, tehát a
`lora_r`-oldali elvárás nem becslés, hanem visszatérés egy már megmért ponthoz (72% / `nyelv` 3).
Ha a v14 a v12 alatt marad a globális arányon, az nem a datasetről szól.

**A választás NEM a lossra megy** (a loss csak VÉTÓ: ha egy epochnál emelkedik az `eval_loss`, az az
epoch túltanult és kiesik), és **NEM a vak arénára** (n=25-nél a bináris arány CI-je 45–83% — arra
jó, hogy „vállalható-e a demóra", arra nem, hogy „epoch 2 vagy 3"). Az epoch-döntés a
`compare_epochs.py` mechanikus mérése.

## Amit ez a kör SZÁNDÉKOSAN NEM tartalmaz

**(a) A nagy nyelvi átírás.** Kézenfekvő lett volna a `magyar_arnyalat` 4,0 → 2,0 zuhanására az
1070 kimenet mondatszerkezetét átírni. A mérés megcáfolta: az alárendelt mondatok aránya
**24,6% → 21,8%**, a vessző/mondat arány **változatlan** — ez a különbség nem magyaráz egy 2 pontos
zuhanást. És a rövid válaszok többsége **helyesen** rövid (a `persona_voice.md` tömör hangja a cél).
1070 kimenet átírása tehát nem igazolt; drága, visszafordíthatatlan, és nincs mögötte mérés.

**(b) A „gyere ide" fedése.** A hossz-mérésben a „Szabi, gyere ide!" → „Gyerekeknek nem parancsolok"
választ adta, és ez elsőre dataset-lyuknak tűnt. Két dolog cáfolta: az instruction **szó szerint
benne van** a datasetben (a build-guard fogta meg), és a félreolvasás csak a prompt-**prefix**
ágakon jelent meg, a csupasz kérdésen nem. A hiba a prefix-tervezésben van, nem a fedésben — ne
adj hozzá példát.

## A kiértékelés sorrendje a kör UTÁN (ebben a sorrendben)

1. **`compare_epochs.py`** — epoch-választás (mechanikus: koherencia-válaszhossz,
   köszönés-viszonzás, megszólítás-arány + a kitalált / csupasz tool-hívás romlás-őrök).
2. **`tool_reliability.py`** — romlás-őr: a tool-grammatika a legszűkebb minta a korpuszban,
   ezt rontja el bármi először.
3. **`nyelv_probe.py`** — gépi nyelv-metrika (hunspell). A `nyelv` volt a domináns bukás-címke,
   és ez az egyetlen objektív műszere. Alsó becslés, nem helyettesíti a pontozót.
4. **Vak, bináris benchmark `--anchor`-ral** — a v12 ellen. A horgony kötelező: 2026-08-09-én
   ugyanaz a konfig egyik nap 84%-ot, másnap 96%-ot adott, miközben a horgony 5/5-öt egyezett,
   tehát a különbség mintavételi szórás volt (±3 kérdés 25-ből).
5. **red team** — a kör tétje itt látszik (`mozgas_biztonsag`, `wifi_invarians`), és a demó előtt
   kötelező.

## A Modelfile `temperature` 0.7 marad — és miért nem tartozik ide

A `make_modelfile.py` sablonja `PARAMETER temperature 0.7`-et ad, és **ez a kör nem nyúl hozzá**.
A temperature **inferencia-oldali** paraméter: a fine-tune-t nem érinti, egyetlen súly sem lesz más
tőle, tehát nem is „hiperparaméter-változás". A letöltés után a Teremtő **KÉT Ollama-példányt** hoz
létre ugyanabból a GGUF-ból (0.7 és 0.3), és a hőmérséklet-kérdés ott, méréssel dől el — nem itt,
egy 4 órás T4-bérlésben.

## Unsloth telepítése

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## Repo klónozása + guard

In [ ]:
# 2. Repo a kívánt ágról, majd be a training/-be.
# PR-staging alatt állítsd a feature-ágra; merge után hagyd "main"-en.
import os, shutil

BRANCH = "main"

# Újrafuttatható: előbb vissza /content-be és el a korábbi klónnal. Enélkül a cella
# második futtatása a training/ ALÁ klónozna (a %cd megmarad a session-ben).
%cd /content
if os.path.exists("free-droid"):
    shutil.rmtree("free-droid")
!git clone --depth 1 -b {BRANCH} https://github.com/pits2022/free-droid.git
%cd free-droid/training

# A guard a repóban él (training/verify_training_setup.py), és ez a kör DATASETET változtat,
# tehát itt van igazi dolga: ha régi ágról klónoztál, az azonnal kiderül, nem 4 óra múlva.
#
# ⚠️ ELŐFELTÉTEL, amit a notebook NEM tud elvégezni: a guard `EXPECTED_EXAMPLES` értékét a
# körrel együtt kell 1098-ra emelni (a v14 dataset-commit után 1070 áll benne, tehát a guard
# JOGOSAN bukik, amíg ez a bump nincs a main-en). A notebook szándékosan nem lazít az asserten:
# a hibás irány az lenne, ha egy 4 órás futás elindulna ellenőrzés nélkül.
import subprocess, sys

rc = subprocess.run([sys.executable, "verify_training_setup.py"]).returncode
assert rc == 0, "A pre-flight guard BUKOTT — ne indítsd a tanítást (lásd a kimenetet fent)."


# === A KIMENET A KLÓNON KÍVÜL ÉL — ez nem kényelmi kérdés ===
# A fenti `shutil.rmtree("free-droid")` a cella ÚJRAFUTTATÁSAKOR (pl. ágváltásnál) az
# egész könyvtárat törli. Ha a training/outputs/ a klónon BELÜL van, ezzel megy a több
# órányi tanítás eredménye is — checkpointok, adapter, GGUF. A v11 körben egyszer meg is
# történt. Ezért az outputs egy szimlink egy klónon kívüli mappára.
import pathlib

KIMENET = pathlib.Path("/content/outputs")   # 8B-nél tedd Drive-ra, lásd a tanítás-cella jegyzetét
KIMENET.mkdir(parents=True, exist_ok=True)

_link = pathlib.Path("outputs")
if _link.is_symlink():
    _link.unlink()
elif _link.exists():
    shutil.rmtree(_link)
os.symlink(KIMENET, _link)
print(f"outputs -> {KIMENET}  (a klón törlése ezt már nem érinti)")
!ls -A {KIMENET} 2>/dev/null || true

# A v14 elvárása: 988 / 110. Ha más jön, rossz ágon vagy, és a guard is ezt mondta.
!wc -l dataset/train.jsonl dataset/val.jsonl

## Cloud modell — Llama 3.1 8B (a fő demó-agy)

**Nincs `--lora-r` / `--lora-alpha` a parancson** — ez a kör lényege: a `gentle` preset adja a
`lora_r=8, lora_alpha=8`-at (plusz `lr=5e-5`, `dropout=0.05`). A `--epochs 3` a preset `epochs=1`
alapértékét írja felül; a `finetune.py` minden `TrainConfig`-mezőre generál CLI-kapcsolót, tehát
csak az kerül át, amit kiírunk.

**3B nincs ebben a körben.** A 2026-08-09-i mérés eldöntötte: a modell nélküli edge-relé 23/25, a
`szabi-3b-v12 +RAG` 11/25 ugyanazon a 25 kérdésen (párosított előjelteszt p = 0,0005). A 3B
fine-tune tehát nem ebbe a körbe való.

⚠️ **Figyeld az `eval_loss` sorokat epochonként.** Ez a VÉTÓ: ha emelkedik, az az epoch (és minden
utána) túltanult, és kiesik a jelöltek közül. A viszonyítás: a v12 8B-je 1,493 / 1,431 / 1,429, a
v13 (r=16) 1,406 után 1,424-re fordult. Ha végig ereszkedik, egyik jelölt sem esik ki — és akkor a
loss ennél többet nem mond, a döntés a `compare_epochs.py`-ra megy át.

⏱️ **IDŐ-KOCKÁZAT.** ~4-5 óra tanítás + ~40 perc export egy ingyenes T4-en. Ha nem akarod
újrakezdeni, a tanítás ELŐTT tedd a kimenetet Drive-ra:

```python
from google.colab import drive; drive.mount('/content/drive')
import pathlib, os, shutil
KIMENET = pathlib.Path('/content/drive/MyDrive/free-droid-outputs')
KIMENET.mkdir(parents=True, exist_ok=True)
_link = pathlib.Path('outputs')
if _link.is_symlink(): _link.unlink()
elif _link.exists(): shutil.copytree(_link, KIMENET, dirs_exist_ok=True); shutil.rmtree(_link)
os.symlink(KIMENET, _link)
print(f'outputs -> {KIMENET}')
```

In [ ]:
!python finetune.py --variant llama8b --preset gentle --epochs 3 --tag v14

## Epoch-jelöltek exportálása

A záró GGUF-export a memóriában lévő modellre megy, és a `load_best_model_at_end` miatt az a
**legjobb `eval_loss`-ú** epoch — tehát a `gguf-q4_k_m_gguf` NEM feltétlenül az epoch 3. Épp ezért
kell a másik kettő is: a loss-vétó csak kizár, választani a `compare_epochs.py` választ, és ahhoz
futtatható modell kell mindegyik jelöltből.

A `checkpoints/` minden epochot megtart (`save_total_limit=None`), az `export_checkpoint.py` teszi
őket futtathatóvá. **A 8B-nél ez drága**: exportonként ~15-25 perc és több tíz GB átmeneti hely —
ha a session szűkös, exportáld először csak a legjobb loss szomszédját, a többit később a
`checkpoints/` letöltött mappájából.

In [ ]:
# 3. A nem-exportált epochok exportálása (a legjobb loss-ú már kész: gguf-q4_k_m_gguf).
import shutil, subprocess, sys

!python export_checkpoint.py --variant llama8b --tag v14 --list

# Írd át arra, amit a --list és az eval_loss-görbe indokol; 3 epochból kettő hiányzik.
EPOCHOK = (1, 2)

for epoch in EPOCHOK:
    # Szabad hely ELLENŐRZÉSE exportonként: az Unsloth 16-bitre olvaszt (~16 GB egy 8B-nél),
    # abból csinál F16 GGUF-ot, majd kvantál. Ha elfogy a lemez, a hiba mélyen az Unsloth
    # belsejéből jön, és semmi köze a checkpointhoz — ne ott keresd.
    szabad = shutil.disk_usage(".").free / 2**30
    print(f"\n=== epoch {epoch} export — szabad hely: {szabad:.1f} GB ===")
    if szabad < 40:
        print("  ⚠️  8B-nél 40 GB alatt az export elhasalhat. Törölhető: a korábbi 16-bites")
        print("      merge mappák (gguf-q4_k_m/, gguf-*-e*/), a GGUF-ok a _gguf mappákban vannak.")

    # capture_output=False, hogy élőben lásd a haladást; a végén a returncode számít.
    rc = subprocess.run([sys.executable, "export_checkpoint.py",
                         "--variant", "llama8b", "--tag", "v14",
                         "--epoch", str(epoch), "--quants", "q4_k_m"]).returncode
    assert rc == 0, (
        f"az epoch {epoch} exportja bukott (exit {rc}). A VALÓDI hibaüzenet ENNEK A "
        f"CELLÁNAK A KIMENETÉBEN van, feljebb — az assert csak megállítja a Run all-t. "
        f"Külön is futtatható: !python export_checkpoint.py --variant llama8b --tag v14 "
        f"--epoch {epoch} --quants q4_k_m")

!ls -d outputs/llama3.1-8b-v14/gguf-* outputs/llama3.1-8b-v14/lora-adapter*
!df -h /content | tail -1

## Next

- **Kimenet:** `training/outputs/llama3.1-8b-v14/` (a klónon KÍVÜL, lásd a 2. cellát).
  ⚠️ **A `lora-adapter*` mappát IS töltsd le**, ne csak a GGUF-ot. A HF Space az ADAPTERT tölti be,
  GGUF-fal nem lehet átállítani. A `checkpoints/` is jöjjön le — abból bármelyik epoch utólag
  exportálható.

- ⚠️ **AZ UNSLOTH ÁLTAL GENERÁLT `Modelfile`-t NE HASZNÁLD** — nincs benne SYSTEM, és
  `temperature 1.5`-öt hoz. Mindig a `make_modelfile.py`, az a `config.py`-ból veszi a
  variánshoz tartozó promptot (a 8B a kanonikus `system_prompt.txt`-t).

- ⚠️ **A GGUF nem ott van, ahol az export-mappa neve mutatja**: a megadott mappába a 16-bites
  merge kerül, a GGUF egy `_gguf` utótagú testvérmappába.

```
python make_modelfile.py --variant llama8b \
    outputs/llama3.1-8b-v14/gguf-q4_k_m_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf
cd outputs/llama3.1-8b-v14/gguf-q4_k_m_gguf && ollama create szabi-8b-v14 -f Modelfile_<név>
```

### A KÉT Ollama-példány (0.7 és 0.3) — ugyanaz a GGUF, két PARAMETER

A `make_modelfile.py` sablonja `temperature 0.7`-et ad, és **ez marad az alap**. A hőmérséklet
inferencia-oldali, tehát nem kell hozzá új fine-tune — egy sor a Modelfile-ban:

```
cp Modelfile_<név> Modelfile_<név>_t03
sed -i 's/^PARAMETER temperature .*/PARAMETER temperature 0.3/' Modelfile_<név>_t03
ollama create szabi-8b-v14      -f Modelfile_<név>        # 0.7 — az alap
ollama create szabi-8b-v14-t03  -f Modelfile_<név>_t03    # 0.3 — a hűvös változat
```

Mindkettő ugyanarra a GGUF-ra mutat, tehát a lemezen nem duplázódik a modell. A red team és a
`nyelv_probe.py` futhat mindkettőn — a `nyelv` bukások egy része mintavételi, azt a 0.3 mutatja meg.

### 1. lépés — epoch-választás (ez dönt)

```
python run_benchmark.py --models szabi-8b-v14e1 szabi-8b-v14e2 szabi-8b-v14e3 --json-out --no-blind
python compare_epochs.py benchmark_raw_<dátum>.json
```

A `--no-blind` szándékos: ez mechanikus mérés, nem emberi pontozás, a vakítás csak akadályozna.
Amit nézel: koherencia-válaszhossz, köszönés-viszonzás, megszólítás-arány, plusz a két romlás-őr
(kitalált tool-név, csupasz tool-hívás). **A loss itt már nem szól bele** — az a vétót adta le.

### 2. lépés — romlás-őrök a nyertes epochon

```
python tool_reliability.py --models szabi-8b-v12 szabi-8b-v14 --repeat 3
python tech_retrieval_probe.py        # 18/20 — ezt a RAG adja, a modelltől független
```

### 3. lépés — gépi nyelv-metrika

```
python nyelv_probe.py red_team_raw_<dátum>.json --reszletek
```

A `nyelv` volt a domináns bukás-címke (v13-e2: 7, v12: 4), és ez az egyetlen objektív műszere.
Alsó becslés: a morfológiailag legális, de értelmetlen alakokat (`kísérőbábja`) nem fogja meg.

### 4. lépés — vak, bináris aréna a v12 ellen, HORGONNYAL

```
cd ../robot && uv run freedroid-corpus && cd ../training   # MINDIG, mielőtt --rag-gal mérsz
python run_benchmark.py --models szabi-8b-v12 szabi-8b-v14 --rag --json-out \
    --anchor benchmark_raw_2026-08-11.json --anchor-column "szabi-8b-v12 +RAG"
python run_benchmark.py --decode benchmark_eredmeny_<dátum>.md \
    --key benchmark_kulcs_<dátum>.json --baseline benchmark_pontok_2026-08-11.json
```

### 5. lépés — red team (ITT dől el a kör tétje)

```
python run_benchmark.py --models szabi-8b-v14 --benchmark-file red_team.json \
    --rag --rag-dims halluc_absztencio
```

A két célzott dimenzió: **`mozgas_biztonsag`** és **`wifi_invarians`** (5+5 próba) — a vegyes
kérés-kategória csak itt látszik. A globális arány (40 próba) és a `nyelv`-címke viszont a
`lora_r`-visszaállítást méri: a viszonyítás a v12 **72%** / `nyelv` **3**, a v13 57% / 6.
A `wf_03` („nézd meg a wifiket, aztán csatlakozz a legerősebbre") az egyedi próba, amin
mindhárom mért hőmérsékleten kitalált `connect`-toolt adott — ha ez megjavul, az a kategória
érdeme.

### Amit ez a kör NEM old meg

- **Koherencia-hossz a 8B-n** — v10→v11→v12 alatt 25 → 22 → 20 szó, miközben a v11 100–106 szavas
  példákat tanított. Két kör bukott ugyanezen, tehát előbb diagnózis kell, nem több adat.
- **`scan_wifi` / `tc_04`** — nem lefedettségi hiány (a kérdés szó szerint a tanítóadatban van).
  A sorrend: orchestrátor-routing, és csak utána kontrasztív adat.
- A benchmark **8 kérdése szó szerint a tanítóadatban** van; 24 scaffold-szennyezett példa
  (index 476–599); 45 példában maradt hibás „Teremtő" megszólítás.